# Week 8 Companion Notebook: HDFS & the Data Lake

## ISM 6562 - Big Data for Business Applications

---

This notebook accompanies the Week 8 lecture slides on **HDFS & the Data Lake**. You will work through HDFS internals hands-on and then apply what you have learned in two complete business scenarios that demonstrate how organizations design, build, and manage data lakes on HDFS.

**What you will do:**

1. Verify your Hadoop cluster and explore the Web UIs
2. Deep-dive into HDFS building blocks: blocks, replication, rack-aware placement, the read path, and CLI commands
3. Explore file format trade-offs (CSV, JSON, and an introduction to Avro)
4. Build a medical analytics data lake for **HealthPulse** (12 hospitals)
5. Build a logistics data lake for **GreenRoute** (2,000 delivery trucks)
6. Reflect on storage trade-offs and design decisions

**Prerequisites:** Your Docker-based Hadoop cluster must be running. Start it with:

```bash
cd week08-hadoop-hdfs
docker-compose up -d
```

---

## Part 0: Setup & Cluster Verification

The `docker-compose.yml` for this week creates a full Hadoop cluster with the following components:

| Component | Container Name | Role |
|---|---|---|
| **NameNode** | `hadoop-namenode` | HDFS metadata, namespace management |
| **DataNode 1** | `hadoop-datanode1` | Block storage |
| **DataNode 2** | `hadoop-datanode2` | Block storage |
| **DataNode 3** | `hadoop-datanode3` | Block storage |
| **ResourceManager** | `hadoop-resourcemanager` | YARN job scheduling |
| **NodeManager 1** | `hadoop-nodemanager1` | YARN compute on node 1 |
| **NodeManager 2** | `hadoop-nodemanager2` | YARN compute on node 2 |
| **NodeManager 3** | `hadoop-nodemanager3` | YARN compute on node 3 |
| **HistoryServer** | `hadoop-historyserver` | MapReduce job history |

**Web UIs** (open in your browser):

- **NameNode UI**: [http://localhost:9870](http://localhost:9870) — browse the filesystem, check DataNode health, view block reports
- **YARN ResourceManager UI**: [http://localhost:8088](http://localhost:8088) — view running/completed applications, node status

Let's verify the cluster is healthy.

In [ ]:
%%bash
# Verify the Hadoop cluster is running
docker exec hadoop-namenode hdfs dfsadmin -report

You should see **3 live DataNodes** (datanode1, datanode2, datanode3) and the configured capacity of the cluster. If you see fewer than 3, wait a minute and re-run — DataNodes may still be starting up.

Also confirm YARN is healthy:

In [ ]:
%%bash
# Check YARN node status
docker exec hadoop-namenode yarn node -list 2>/dev/null || echo "(YARN may still be starting)"
echo ""
# Quick check: can we list the HDFS root?
docker exec hadoop-namenode hdfs dfs -ls /

---

## Part 1: Concept Review — HDFS Building Blocks

Before we dive into the business scenarios, let’s do a hands-on exploration of how HDFS actually stores, replicates, and serves data. This section maps directly to Sections 3 and 4 of the lecture slides.

### 1a. Blocks and Replication

Every file in HDFS is split into **blocks** (default 128 MB). Each block is replicated across multiple DataNodes for fault tolerance. The **replication factor (RF)** determines how many copies exist.

Let’s upload a small file and inspect how HDFS stores it.

In [ ]:
%%bash
# Create a test directory and upload a small file
docker exec hadoop-namenode bash -c '
    hdfs dfs -mkdir -p /concept-review

    # Create a small text file
    echo "This is a test file to explore HDFS building blocks." > /tmp/test_block.txt
    echo "HDFS will store this as one block because it is much smaller than 128MB." >> /tmp/test_block.txt
    echo "Replication ensures copies exist on multiple DataNodes." >> /tmp/test_block.txt

    hdfs dfs -put -f /tmp/test_block.txt /concept-review/

    echo "File uploaded successfully."
    hdfs dfs -ls -h /concept-review/
'

In [ ]:
%%bash
# Check block info with hdfs fsck
docker exec hadoop-namenode bash -c '
    echo "=== BLOCK INFO ==="
    hdfs fsck /concept-review/test_block.txt -files -blocks -locations
'

Notice in the output above:

- The file is stored as **1 block** (it is much smaller than the 128MB block size)
- The block is **replicated** across DataNodes (the default replication factor is 2 in our cluster)
- The `fsck` output shows exactly which DataNodes hold each replica

In [ ]:
%%bash
# Demonstrate replication factor and file metadata with hdfs dfs -stat
docker exec hadoop-namenode bash -c '
    echo "=== FILE METADATA ==="
    echo "Replication factor: $(hdfs dfs -stat %r /concept-review/test_block.txt)"
    echo "Block size (bytes): $(hdfs dfs -stat %o /concept-review/test_block.txt)"
    echo "File size (bytes):  $(hdfs dfs -stat %b /concept-review/test_block.txt)"
    echo "Last modified:      $(hdfs dfs -stat %y /concept-review/test_block.txt)"
    echo "Owner:              $(hdfs dfs -stat %u /concept-review/test_block.txt)"
    echo "Group:              $(hdfs dfs -stat %g /concept-review/test_block.txt)"
'

### Quick Exercise: Block Prediction

The default HDFS block size is **128 MB**.

**Question:** If you upload a **500 MB** file to HDFS, how many blocks will it be split into?

Think about it, then run the cell below to check your answer.

In [ ]:
import math

file_size_mb = 500
block_size_mb = 128

num_blocks = math.ceil(file_size_mb / block_size_mb)

print(f"A {file_size_mb} MB file with a {block_size_mb} MB block size requires: {num_blocks} blocks")
print()
print("Breakdown:")
for i in range(num_blocks):
    start = i * block_size_mb
    end = min((i + 1) * block_size_mb, file_size_mb)
    print(f"  Block {i+1}: {start} MB - {end} MB  ({end - start} MB)")
print()
print("The last block is only 116 MB — HDFS does NOT pad it to 128 MB.")
print("This is different from traditional filesystems that allocate full block sizes.")

### 1b. Rack-Aware Replication Walkthrough

In a production Hadoop cluster, the **NameNode** decides where to place each block replica using a **rack topology map**. The placement algorithm works like this:

1. The **client** asks the NameNode to create a file.
2. The NameNode consults its rack topology map and returns a **pipeline list** of DataNodes — for example `[DataNode-A, DataNode-D, DataNode-E]`.
3. The client sends data to the **first** DataNode in the pipeline.
4. Each DataNode in the pipeline **forwards** the data to the next hop. DataNodes never contact the NameNode during writes.
5. Acknowledgments flow back up the pipeline to the client.

With **RF=3** and **2 racks**, the default policy places:

- 1st replica on a node in the **local rack** (same rack as the writer)
- 2nd replica on a node in a **remote rack**
- 3rd replica on a **different node** in the remote rack

This means a single rack failure still leaves 1 copy available, while keeping 2 copies on the same rack for fast local reads.

**How does the NameNode know the rack layout?** An administrator configures either a topology script or a static mapping table that maps IP addresses to rack identifiers.

Our Docker cluster has all nodes on the same virtual network (effectively one rack), but the HDFS commands work exactly the same. Let’s see replication in action.

In [ ]:
%%bash
# Upload a file and inspect its replica placement
docker exec hadoop-namenode bash -c '
    # Create a file with enough content to see replication clearly
    for i in $(seq 1 5000); do
        echo "Line $i: This is sample data for demonstrating HDFS replication placement across DataNodes."
    done > /tmp/replication_demo.txt

    hdfs dfs -put -f /tmp/replication_demo.txt /concept-review/

    echo "=== Replication Factor ==="
    echo "RF: $(hdfs dfs -stat %r /concept-review/replication_demo.txt)"
    echo ""

    echo "=== Block Locations (which DataNodes hold replicas?) ==="
    hdfs fsck /concept-review/replication_demo.txt -files -blocks -locations
'

In the `fsck` output, each block line shows its replicas in the format `[DataNode:port]`. With RF=2, you should see 2 DataNode locations per block. In a multi-rack production cluster, these would be spread across racks according to the rack-aware placement policy.

### 1c. Read Path Demonstration

The **HDFS read path** is different from the write path:

1. The client contacts the **NameNode** and asks for the block locations of a file.
2. The NameNode returns metadata: which blocks, which DataNodes hold each block, sorted by proximity.
3. The client reads **directly from the DataNodes** — the NameNode is not involved in the actual data transfer.

This is a critical design decision: the NameNode handles metadata only, which keeps it from becoming a bottleneck. The actual data flows directly between clients and DataNodes.

Let’s create a multi-block file to see this more clearly.

In [ ]:
%%bash
# Generate a ~300MB file to create multiple HDFS blocks
docker exec hadoop-namenode bash -c '
    echo "Generating ~300MB file (this may take a moment)..."

    # Generate ~300MB of data using dd + base64 for speed
    dd if=/dev/urandom bs=1M count=300 2>/dev/null | base64 > /tmp/multiblock_demo.txt 2>/dev/null

    actual_size=$(ls -lh /tmp/multiblock_demo.txt | awk "{print \$5}")
    echo "Generated file size: $actual_size"
    echo ""

    echo "Uploading to HDFS..."
    hdfs dfs -put -f /tmp/multiblock_demo.txt /concept-review/
    echo "Upload complete."
'

In [ ]:
%%bash
# Show block locations for the multi-block file
docker exec hadoop-namenode bash -c '
    echo "=== FILE METADATA ==="
    echo "File size:    $(hdfs dfs -stat %b /concept-review/multiblock_demo.txt) bytes"
    echo "Block size:   $(hdfs dfs -stat %o /concept-review/multiblock_demo.txt) bytes"
    echo "Replication:  $(hdfs dfs -stat %r /concept-review/multiblock_demo.txt)"
    echo ""

    echo "=== BLOCK LOCATIONS ==="
    echo "Each block below shows which DataNodes hold its replicas."
    echo "When a client reads this file, it contacts the NameNode ONCE to get"
    echo "this metadata, then reads each block DIRECTLY from a DataNode."
    echo ""
    hdfs fsck /concept-review/multiblock_demo.txt -files -blocks -locations
'

Notice there are **multiple blocks** (a ~300MB file at 128MB block size = 3 blocks). Each block is independently replicated across DataNodes.

When you run `hdfs dfs -cat`, the client:

1. Asks the NameNode for block locations (lightweight metadata call)
2. Reads Block 0 directly from one of its DataNode replicas
3. Reads Block 1 directly from one of its DataNode replicas
4. Reads Block 2 directly from one of its DataNode replicas

The NameNode never touches the actual data. This is why HDFS scales — you can add more DataNodes for more throughput without overloading the NameNode.

### 1d. HDFS CLI Deep Dive

The HDFS command-line interface mirrors many familiar Linux commands. Let’s walk through the key commands from the lecture slides.

In [ ]:
%%bash
# HDFS CLI — Basic file operations
docker exec hadoop-namenode bash -c '
    echo "=== mkdir: Create directories ==="
    hdfs dfs -mkdir -p /concept-review/cli-demo/subdir1/nested
    hdfs dfs -mkdir -p /concept-review/cli-demo/subdir2
    echo "Directories created."
    echo ""

    echo "=== ls: List directory contents ==="
    hdfs dfs -ls /concept-review/cli-demo/
    echo ""

    echo "=== put: Upload a local file to HDFS ==="
    echo "Hello from HDFS CLI demo" > /tmp/cli_test.txt
    echo "Second line of the file" >> /tmp/cli_test.txt
    hdfs dfs -put /tmp/cli_test.txt /concept-review/cli-demo/subdir1/
    echo "File uploaded."
    echo ""

    echo "=== cat: Read a file from HDFS ==="
    hdfs dfs -cat /concept-review/cli-demo/subdir1/cli_test.txt
    echo ""

    echo "=== get: Download a file from HDFS to local ==="
    hdfs dfs -get /concept-review/cli-demo/subdir1/cli_test.txt /tmp/downloaded.txt
    echo "Downloaded. Contents:"
    cat /tmp/downloaded.txt
    echo ""

    echo "=== stat: View file metadata ==="
    echo "Size (bytes):  $(hdfs dfs -stat %b /concept-review/cli-demo/subdir1/cli_test.txt)"
    echo "Block size:    $(hdfs dfs -stat %o /concept-review/cli-demo/subdir1/cli_test.txt)"
    echo "Replication:   $(hdfs dfs -stat %r /concept-review/cli-demo/subdir1/cli_test.txt)"
    echo "Modified:      $(hdfs dfs -stat %y /concept-review/cli-demo/subdir1/cli_test.txt)"
'

In [ ]:
%%bash
# HDFS CLI — Storage and monitoring commands
docker exec hadoop-namenode bash -c '
    echo "=== du: Disk usage ==="
    echo "Logical size (without replication):"
    hdfs dfs -du -s -h /concept-review/
    echo ""
    echo "Breakdown by file:"
    hdfs dfs -du -h /concept-review/
    echo ""

    echo "=== setrep: Change replication factor ==="
    echo "Before: RF=$(hdfs dfs -stat %r /concept-review/cli-demo/subdir1/cli_test.txt)"
    hdfs dfs -setrep 3 /concept-review/cli-demo/subdir1/cli_test.txt
    echo "After:  RF=$(hdfs dfs -stat %r /concept-review/cli-demo/subdir1/cli_test.txt)"
    echo ""

    echo "=== dfsadmin -report: Cluster health summary ==="
    hdfs dfsadmin -report 2>/dev/null | head -20
    echo "... (truncated)"
    echo ""

    echo "=== fsck: Filesystem health check ==="
    hdfs fsck /concept-review -files 2>/dev/null | tail -15
'

In [ ]:
%%bash
# HDFS CLI — rm: Remove files
docker exec hadoop-namenode bash -c '
    echo "=== rm: Remove a file ==="
    hdfs dfs -rm /concept-review/cli-demo/subdir1/cli_test.txt
    echo ""

    echo "=== rm -r: Remove a directory recursively ==="
    hdfs dfs -rm -r /concept-review/cli-demo
    echo ""

    echo "Remaining contents of /concept-review/:"
    hdfs dfs -ls /concept-review/
'

### 1e. File Format Exploration

The lecture slides cover four file formats: **CSV**, **JSON**, **Apache Avro**, and **Apache Parquet**. Before we get to the business scenarios, let’s see the size difference between the text-based formats (CSV and JSON) using the same dataset.

| Format | Type | Schema | Best For |
|--------|------|--------|----------|
| **CSV** | Text, row-oriented | In header row | Simple tabular data, human-readable |
| **JSON** | Text, row-oriented | Self-describing per record | Nested/semi-structured data, APIs |
| **Avro** | Binary, row-oriented | Embedded in file header | Streaming ingest, schema evolution, Kafka |
| **Parquet** | Binary, columnar | Embedded in file footer | Analytics queries, column projections |

The key insight from the slides: **Avro for ingest, Parquet for analytics.** Both are binary formats that dramatically reduce storage compared to CSV/JSON, but they optimize for different access patterns.

In [ ]:
%%bash
# Compare CSV vs JSON size for the same dataset
docker exec hadoop-namenode bash -c '
    # Create a small inline dataset to illustrate the size difference
    # (10 records are enough to demonstrate the pattern)

    cat > /tmp/format_demo.csv << "CSVEOF"
sensor_id,timestamp,temperature,humidity,pressure,status
S-042,2026-04-01T00:00:00,78.31,45.2,1013.5,normal
S-007,2026-04-01T00:01:40,92.14,62.8,1001.3,warning
S-099,2026-04-01T00:03:20,65.87,33.1,1045.2,normal
S-023,2026-04-01T00:05:00,101.44,88.7,985.6,critical
S-051,2026-04-01T00:06:40,71.09,55.0,1022.1,normal
S-015,2026-04-01T00:08:20,84.56,41.9,1008.7,normal
S-088,2026-04-01T00:10:00,97.22,72.3,994.1,warning
S-031,2026-04-01T00:11:40,68.93,29.4,1038.9,normal
S-076,2026-04-01T00:13:20,89.17,58.6,1005.4,normal
S-004,2026-04-01T00:15:00,73.48,47.5,1019.8,normal
CSVEOF

    cat > /tmp/format_demo.json << "JSONEOF"
{"sensor_id": "S-042", "timestamp": "2026-04-01T00:00:00", "temperature": 78.31, "humidity": 45.2, "pressure": 1013.5, "status": "normal"}
{"sensor_id": "S-007", "timestamp": "2026-04-01T00:01:40", "temperature": 92.14, "humidity": 62.8, "pressure": 1001.3, "status": "warning"}
{"sensor_id": "S-099", "timestamp": "2026-04-01T00:03:20", "temperature": 65.87, "humidity": 33.1, "pressure": 1045.2, "status": "normal"}
{"sensor_id": "S-023", "timestamp": "2026-04-01T00:05:00", "temperature": 101.44, "humidity": 88.7, "pressure": 985.6, "status": "critical"}
{"sensor_id": "S-051", "timestamp": "2026-04-01T00:06:40", "temperature": 71.09, "humidity": 55.0, "pressure": 1022.1, "status": "normal"}
{"sensor_id": "S-015", "timestamp": "2026-04-01T00:08:20", "temperature": 84.56, "humidity": 41.9, "pressure": 1008.7, "status": "normal"}
{"sensor_id": "S-088", "timestamp": "2026-04-01T00:10:00", "temperature": 97.22, "humidity": 72.3, "pressure": 994.1, "status": "warning"}
{"sensor_id": "S-031", "timestamp": "2026-04-01T00:11:40", "temperature": 68.93, "humidity": 29.4, "pressure": 1038.9, "status": "normal"}
{"sensor_id": "S-076", "timestamp": "2026-04-01T00:13:20", "temperature": 89.17, "humidity": 58.6, "pressure": 1005.4, "status": "normal"}
{"sensor_id": "S-004", "timestamp": "2026-04-01T00:15:00", "temperature": 73.48, "humidity": 47.5, "pressure": 1019.8, "status": "normal"}
JSONEOF

    echo "=== SIZE COMPARISON (10 identical sensor records) ==="
    csv_size=$(stat -c%s /tmp/format_demo.csv)
    json_size=$(stat -c%s /tmp/format_demo.json)
    echo "CSV:  $csv_size bytes"
    echo "JSON: $json_size bytes"
    echo ""
    echo "JSON is ~$(( (json_size - csv_size) * 100 / csv_size ))% larger than CSV for the same data."
    echo "This is because JSON repeats field names in every record."
    echo ""
    echo "=== What about Avro? ==="
    echo "Avro is a binary format that stores the schema ONCE in the file header"
    echo "and encodes each record as compact binary. For this dataset, Avro would"
    echo "be roughly 3-4x smaller than JSON:"
    echo "  - No repeated field names (schema stored once)"
    echo "  - Binary encoding for numbers (4-8 bytes vs 5-10 chars)"
    echo "  - No delimiters, quotes, or whitespace overhead"
    echo ""
    echo "At 10,000 records, the estimated sizes would be:"
    echo "  CSV:  ~$(( csv_size * 1000 / 1024 )) KB"
    echo "  JSON: ~$(( json_size * 1000 / 1024 )) KB"
    echo "  Avro: ~$(( json_size * 1000 / 1024 / 3 )) KB (estimated ~3x smaller than JSON)"
    echo ""
    echo "=== Sample records ==="
    echo "CSV (first 3 lines):"
    head -3 /tmp/format_demo.csv
    echo ""
    echo "JSON (first 2 records):"
    head -2 /tmp/format_demo.json
'

**Key takeaways:**

- **CSV** is compact for flat data but cannot represent nested structures.
- **JSON** handles nested data naturally but is verbose — field names repeat in every record.
- **Avro** (binary, row-oriented) stores the schema once and encodes data compactly. Ideal for **write-heavy ingest** and **streaming** (Kafka’s native format).
- **Parquet** (binary, columnar) is ideal for **read-heavy analytics** where queries only need a few columns.

In the business scenarios below, we will see exactly when and why you would choose each format.

In [ ]:
%%bash
# Clean up the concept review directory
docker exec hadoop-namenode bash -c '
    hdfs dfs -rm -r /concept-review
    rm -f /tmp/test_block.txt /tmp/replication_demo.txt /tmp/multiblock_demo.txt
    rm -f /tmp/cli_test.txt /tmp/downloaded.txt
    rm -f /tmp/format_demo.csv /tmp/format_demo.json
'
echo "Concept review cleanup complete." 

---

## Part 2: Business Scenario 1 — HealthPulse Medical Analytics

### The Story

**HealthPulse** is a healthcare analytics company that serves **12 hospitals** across the southeastern United States. Each hospital has its own IT systems, its own data formats, and its own quirks. The Chief Data Officer has a mandate:

> "We need a centralized data lake where we can run cross-hospital analytics — readmission risk prediction, resource utilization, and insurance claim optimization. But here’s the catch: each hospital’s data looks different. We can’t force them to standardize before sending us data."

This scenario demonstrates **schema-on-read**, **replication strategies for compliance (HIPAA)**, and the use of **different file formats** for different data types.

### Step 2.1: Design the Directory Structure

A well-designed data lake follows the **landing / curated / analytics** zone pattern:

- **Landing zone**: Raw data exactly as received from each source (schema-on-read)
- **Curated zone**: Cleaned, standardized data ready for analysis
- **Analytics zone**: Aggregated results and model outputs

```
/healthpulse-lake/
├── landing/patient-records/hospital-{01..12}/
├── landing/device-readings/2026/04/{01..30}/
├── landing/insurance-claims/
├── landing/lab-results/
├── curated/unified-patients/
├── curated/standardized-claims/
├── analytics/readmission-risk/
└── analytics/resource-utilization/
```

In [ ]:
%%bash
# Create the HealthPulse data lake directory structure
docker exec hadoop-namenode bash -c '
    echo "Creating HealthPulse data lake structure..."

    # Landing zone — patient records for all 12 hospitals
    for i in $(seq -w 1 12); do
        hdfs dfs -mkdir -p /healthpulse-lake/landing/patient-records/hospital-$i
    done

    # Landing zone — device readings partitioned by date
    for day in $(seq -w 1 30); do
        hdfs dfs -mkdir -p /healthpulse-lake/landing/device-readings/2026/04/$day
    done

    # Landing zone — other sources
    hdfs dfs -mkdir -p /healthpulse-lake/landing/insurance-claims
    hdfs dfs -mkdir -p /healthpulse-lake/landing/lab-results

    # Curated zone
    hdfs dfs -mkdir -p /healthpulse-lake/curated/unified-patients
    hdfs dfs -mkdir -p /healthpulse-lake/curated/standardized-claims

    # Analytics zone
    hdfs dfs -mkdir -p /healthpulse-lake/analytics/readmission-risk
    hdfs dfs -mkdir -p /healthpulse-lake/analytics/resource-utilization

    echo ""
    echo "=== HealthPulse Data Lake Structure ==="
    hdfs dfs -ls -R /healthpulse-lake | head -40
    echo "... (truncated — showing first 40 entries)"
    echo ""
    echo "Total directories created: $(hdfs dfs -ls -R /healthpulse-lake | wc -l)"
'

### Step 2.2: Download & Load Patient Records (CSV)

Here is where schema-on-read comes to life. Each hospital sends patient records, but their schemas are **different**:

- **Hospital-01** uses the column name `dept` instead of `department`
- **Hospital-02** uses `department` (the "correct" name)
- **Hospital-03** includes extra columns (`insurance_id`, `attending_physician`) that the others do not have

In a traditional relational database, you would need to force all hospitals into one schema before loading. In a data lake, you **accept the data as-is** and reconcile schemas at read time.

The pre-generated patient record CSVs (10,000 records each) are available in the course repository under `week08-hadoop-hdfs/data/healthpulse/patient-records/`. If you want to see how the data was created, look at `data/generate_all.py` in the repo.

In [ ]:
%%bash
# Download pre-generated patient records from the course repo and upload to HDFS
docker exec hadoop-namenode bash -c '
    REPO_BASE="https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main/week08/data"

    echo "Downloading patient records from course repository..."
    echo ""

    curl -sL "$REPO_BASE/healthpulse/patient-records/hospital_01_patients.csv" -o /tmp/hospital_01_patients.csv
    curl -sL "$REPO_BASE/healthpulse/patient-records/hospital_02_patients.csv" -o /tmp/hospital_02_patients.csv
    curl -sL "$REPO_BASE/healthpulse/patient-records/hospital_03_patients.csv" -o /tmp/hospital_03_patients.csv

    echo "=== Schema differences across hospitals ==="
    echo ""
    echo "Hospital-01 header (uses dept):"
    head -1 /tmp/hospital_01_patients.csv
    echo ""
    echo "Hospital-02 header (uses department):"
    head -1 /tmp/hospital_02_patients.csv
    echo ""
    echo "Hospital-03 header (extra columns: insurance_id, attending_physician):"
    head -1 /tmp/hospital_03_patients.csv
    echo ""

    echo "Sample records from each:"
    echo "  Hospital-01: $(head -2 /tmp/hospital_01_patients.csv | tail -1)"
    echo "  Hospital-02: $(head -2 /tmp/hospital_02_patients.csv | tail -1)"
    echo "  Hospital-03: $(head -2 /tmp/hospital_03_patients.csv | tail -1)"
    echo ""

    echo "File sizes:"
    ls -lh /tmp/hospital_0*_patients.csv
'

In [ ]:
%%bash
# Upload patient records to HDFS landing zone
docker exec hadoop-namenode bash -c '
    hdfs dfs -put /tmp/hospital_01_patients.csv /healthpulse-lake/landing/patient-records/hospital-01/
    hdfs dfs -put /tmp/hospital_02_patients.csv /healthpulse-lake/landing/patient-records/hospital-02/
    hdfs dfs -put /tmp/hospital_03_patients.csv /healthpulse-lake/landing/patient-records/hospital-03/

    echo "=== Patient records uploaded to HDFS ==="
    echo ""
    echo "Hospital-01:"
    hdfs dfs -ls -h /healthpulse-lake/landing/patient-records/hospital-01/
    echo ""
    echo "Hospital-02:"
    hdfs dfs -ls -h /healthpulse-lake/landing/patient-records/hospital-02/
    echo ""
    echo "Hospital-03:"
    hdfs dfs -ls -h /healthpulse-lake/landing/patient-records/hospital-03/
    echo ""
    echo "Verifying row counts:"
    echo "  Hospital-01: $(hdfs dfs -cat /healthpulse-lake/landing/patient-records/hospital-01/hospital_01_patients.csv | wc -l) rows (including header)"
    echo "  Hospital-02: $(hdfs dfs -cat /healthpulse-lake/landing/patient-records/hospital-02/hospital_02_patients.csv | wc -l) rows (including header)"
    echo "  Hospital-03: $(hdfs dfs -cat /healthpulse-lake/landing/patient-records/hospital-03/hospital_03_patients.csv | wc -l) rows (including header)"
'

### Step 2.3: Download & Load Medical Device Readings

Hospitals generate massive amounts of IoT data from bedside monitors. This data is naturally **nested** — a blood pressure reading has both systolic and diastolic values, a heart monitor has BPM and rhythm.

**Format choice: Avro in production, JSON for this demo.**

In a real HealthPulse deployment, device readings would be stored in **Apache Avro** format for several reasons:

- **Schema evolution**: When hospitals add new device types or new sensors, Avro handles schema changes gracefully (new fields get defaults, removed fields are ignored).
- **Compact binary encoding**: Avro stores the schema once in the file header and encodes each record as compact binary — roughly **3x smaller** than JSON.
- **Kafka-native**: If HealthPulse streams device data through Kafka before landing in HDFS, Avro is Kafka's native serialization format.

For this notebook, we use JSON so you can inspect the data visually. The pre-generated file contains 50,000 nested device readings. See `data/generate_all.py` in the repo for the generation script. We will calculate the Avro size savings to understand the production trade-off.

In [ ]:
%%bash
# Download pre-generated medical device readings from the course repo
docker exec hadoop-namenode bash -c '
    REPO_BASE="https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main/week08/data"

    echo "Downloading 50,000 medical device readings..."
    curl -sL "$REPO_BASE/healthpulse/device-readings/device_readings.json" -o /tmp/device_readings.json

    echo "File size:"
    ls -lh /tmp/device_readings.json
    echo ""
    echo "Record count: $(wc -l < /tmp/device_readings.json) records"
    echo ""
    echo "Sample records (first 3, pretty-printed):"
    head -3 /tmp/device_readings.json | python3 -m json.tool --no-ensure-ascii
'

In [ ]:
%%bash
# Upload device readings to HDFS
docker exec hadoop-namenode bash -c '
    hdfs dfs -put /tmp/device_readings.json /healthpulse-lake/landing/device-readings/2026/04/01/

    echo "=== Device readings uploaded ==="
    hdfs dfs -ls -h /healthpulse-lake/landing/device-readings/2026/04/01/
    echo ""
    echo "Notice: this nested JSON contains blood_pressure.systolic, heart_monitor.bpm, etc."
    echo "A relational database would need separate tables or flattened columns for each device type."
    echo "HDFS stores it as-is — schema-on-read handles the rest."
'

In [ ]:
# Calculate Avro size savings for HealthPulse device readings

print("=" * 65)
print("   HEALTHPULSE — AVRO vs JSON SIZE COMPARISON")
print("=" * 65)
print()

# We know the JSON file characteristics from the generation above
num_readings = 50000
avg_json_bytes_per_record = 180  # typical for these nested device records
json_total = num_readings * avg_json_bytes_per_record

# Avro encoding estimates:
# - Schema stored once in header (~500 bytes)
# - Each record: device_id (~12 bytes), patient_id (~12 bytes),
#   timestamp (~8 bytes as long), device_type (~1 byte as enum index),
#   reading fields (~12-20 bytes as binary ints/floats + short strings)
# - Total per record: ~50-60 bytes
avg_avro_bytes_per_record = 55
avro_total = (num_readings * avg_avro_bytes_per_record) + 500  # +500 for header

ratio = json_total / avro_total
savings_pct = (1 - avro_total / json_total) * 100

print(f"Records:              {num_readings:,}")
print(f"JSON size (est):      {json_total / (1024*1024):.1f} MB  ({avg_json_bytes_per_record} bytes/record)")
print(f"Avro size (est):      {avro_total / (1024*1024):.1f} MB  ({avg_avro_bytes_per_record} bytes/record)")
print(f"Compression ratio:    {ratio:.1f}x")
print(f"Storage savings:      {savings_pct:.0f}%")
print()
print("Why Avro for device readings in production:")
print("  1. Schema evolution — new device types (e.g., ventilator) can be")
print("     added without breaking existing readers")
print("  2. Compact binary — ~3x smaller than JSON at 50K readings")
print("  3. Kafka-native — if streaming device data through Kafka,")
print("     Avro serialization is the standard choice")
print("  4. Self-describing — schema is embedded in the file, unlike CSV")

### Step 2.4: Download & Load Insurance Claims (CSV)

Insurance claims are critical business data — they drive revenue for the hospitals and are subject to audit. Every claim must be traceable. The pre-generated file contains 20,000 claims across all three hospitals.

In [ ]:
%%bash
# Download pre-generated insurance claims from the course repo
docker exec hadoop-namenode bash -c '
    REPO_BASE="https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main/week08/data"

    echo "Downloading 20,000 insurance claims..."
    curl -sL "$REPO_BASE/healthpulse/insurance-claims/insurance_claims.csv" -o /tmp/insurance_claims.csv

    echo "File size:"
    ls -lh /tmp/insurance_claims.csv
    echo ""
    echo "Header:"
    head -1 /tmp/insurance_claims.csv
    echo ""
    echo "Sample records:"
    head -4 /tmp/insurance_claims.csv | tail -3
    echo ""
    echo "Row count: $(wc -l < /tmp/insurance_claims.csv) (including header)"
'

In [ ]:
%%bash
# Upload insurance claims to HDFS
docker exec hadoop-namenode bash -c '
    hdfs dfs -put /tmp/insurance_claims.csv /healthpulse-lake/landing/insurance-claims/

    echo "=== Insurance claims uploaded ==="
    hdfs dfs -ls -h /healthpulse-lake/landing/insurance-claims/
'

### Step 2.5: Set Replication Strategies

Not all data is equal. In healthcare, **patient records** are subject to HIPAA compliance and must be highly available — losing them would be catastrophic. **Device readings**, on the other hand, are high-volume and can often be re-ingested from the devices if needed.

We can set different replication factors for different data based on criticality:

- **RF=3** for patient records (critical, HIPAA compliance)
- **RF=2** for device readings (high volume, re-ingestable)

**File format strategy:**

| Data Source | Format | Why |
|---|---|---|
| Patient records | **CSV** | Flat tabular data from hospital EHR exports |
| Device readings | **Avro** (JSON in demo) | Schema evolution for new devices, compact binary, Kafka-ready |
| Insurance claims | **CSV** | Flat claims data from billing systems |
| Curated/Analytics | **Parquet** | Columnar format for fast analytical queries |

In [ ]:
%%bash
# Set replication strategies based on data criticality
docker exec hadoop-namenode bash -c '
    echo "=== Setting Replication Factors ==="
    echo ""

    # Patient records: RF=3 (critical, HIPAA)
    echo "Setting RF=3 for patient records (HIPAA compliance)..."
    hdfs dfs -setrep -R 3 /healthpulse-lake/landing/patient-records/
    echo ""

    # Device readings: RF=2 (high volume, can be re-ingested)
    echo "Setting RF=2 for device readings (high volume)..."
    hdfs dfs -setrep -R 2 /healthpulse-lake/landing/device-readings/
    echo ""

    # Insurance claims: keep default RF=2 (cluster default)
    echo "Insurance claims: keeping cluster default RF=2"
    echo ""

    echo "=== Verifying Replication Factors ==="
    echo ""
    echo "Patient records (hospital-01):"
    hdfs dfs -stat "  File: %n | Replication: %r" /healthpulse-lake/landing/patient-records/hospital-01/hospital_01_patients.csv
    echo ""
    echo "Patient records (hospital-02):"
    hdfs dfs -stat "  File: %n | Replication: %r" /healthpulse-lake/landing/patient-records/hospital-02/hospital_02_patients.csv
    echo ""
    echo "Device readings:"
    hdfs dfs -stat "  File: %n | Replication: %r" /healthpulse-lake/landing/device-readings/2026/04/01/device_readings.json
    echo ""
    echo "Insurance claims:"
    hdfs dfs -stat "  File: %n | Replication: %r" /healthpulse-lake/landing/insurance-claims/insurance_claims.csv
'

In [ ]:
%%bash
# Verify block-level replication with fsck
docker exec hadoop-namenode bash -c '
    echo "=== Block-level verification for patient records ==="
    hdfs fsck /healthpulse-lake/landing/patient-records/hospital-01/hospital_01_patients.csv -files -blocks -locations
'

### Step 2.6: Verify & Report

Let’s generate a comprehensive data lake inventory — the kind of report you would present to the CDO showing the state of the HealthPulse data lake.

In [ ]:
%%bash
# Generate a full data lake inventory report
docker exec hadoop-namenode bash -c '
    echo "================================================================"
    echo "       HEALTHPULSE DATA LAKE — INVENTORY REPORT"
    echo "================================================================"
    echo ""

    echo "LANDING ZONE — Patient Records"
    echo "--------------------------------"
    for i in 01 02 03; do
        count=$(hdfs dfs -cat /healthpulse-lake/landing/patient-records/hospital-$i/*.csv 2>/dev/null | tail -n +2 | wc -l)
        size=$(hdfs dfs -du -s -h /healthpulse-lake/landing/patient-records/hospital-$i 2>/dev/null | awk "{print \$1, \$2}")
        rf=$(hdfs dfs -stat %r /healthpulse-lake/landing/patient-records/hospital-$i/*.csv 2>/dev/null)
        echo "  Hospital-$i: $count records | Size: $size | RF: $rf"
    done
    echo ""

    echo "LANDING ZONE — Device Readings"
    echo "--------------------------------"
    dev_count=$(hdfs dfs -cat /healthpulse-lake/landing/device-readings/2026/04/01/device_readings.json 2>/dev/null | wc -l)
    dev_size=$(hdfs dfs -du -s -h /healthpulse-lake/landing/device-readings 2>/dev/null | awk "{print \$1, \$2}")
    dev_rf=$(hdfs dfs -stat %r /healthpulse-lake/landing/device-readings/2026/04/01/device_readings.json 2>/dev/null)
    echo "  Readings: $dev_count records | Size: $dev_size | RF: $dev_rf"
    echo "  Format: JSON (production: Avro for schema evolution + compact binary)"
    echo ""

    echo "LANDING ZONE — Insurance Claims"
    echo "--------------------------------"
    claim_count=$(hdfs dfs -cat /healthpulse-lake/landing/insurance-claims/insurance_claims.csv 2>/dev/null | tail -n +2 | wc -l)
    claim_size=$(hdfs dfs -du -s -h /healthpulse-lake/landing/insurance-claims 2>/dev/null | awk "{print \$1, \$2}")
    claim_rf=$(hdfs dfs -stat %r /healthpulse-lake/landing/insurance-claims/insurance_claims.csv 2>/dev/null)
    echo "  Claims: $claim_count records | Size: $claim_size | RF: $claim_rf"
    echo ""

    echo "STORAGE SUMMARY"
    echo "--------------------------------"
    echo "  Total logical size (without replication):"
    hdfs dfs -du -s -h /healthpulse-lake/landing 2>/dev/null | awk "{print \"    \", \$1, \$2}"
    echo ""
    echo "  Storage by zone:"
    hdfs dfs -du -h /healthpulse-lake/landing 2>/dev/null
    echo ""

    echo "BLOCK DISTRIBUTION"
    echo "--------------------------------"
    hdfs fsck /healthpulse-lake/landing -files 2>/dev/null | grep -E "Total (blocks|size)|Minimally|Over-replicated|Under-replicated|Default replication"
    echo ""

    echo "================================================================"
    echo "       HEALTHPULSE DATA LAKE — OPERATIONAL"
    echo "================================================================"
'

---

## Part 3: Business Scenario 2 — GreenRoute Logistics

### The Story

**GreenRoute** operates a fleet of **2,000 delivery trucks** across the United States. Their VP of Operations has a vision:

> "Every truck generates GPS coordinates every 10 seconds. That’s 8,640 readings per truck per day. Multiply by 2,000 trucks and we’re looking at 17 million GPS records daily. Add in delivery receipts, weather data, and maintenance logs — we need infrastructure that can handle this volume and let us optimize routes, predict delivery times, and reduce fuel costs."

This scenario demonstrates HDFS’s ability to handle **high-volume time-series data** with **temporal partitioning**, **storage tiering**, and the **Avro-to-Parquet pipeline pattern**.

### Step 3.1: Design the Directory Structure

GreenRoute’s data is heavily **time-partitioned**. GPS telemetry is partitioned down to the **hour** — this means a query for "all truck positions at 2 PM on April 3rd" only needs to read one hourly partition instead of scanning the entire dataset.

```
/greenroute-lake/
├── landing/gps-telemetry/2026/04/{01..07}/{00..23}/
├── landing/inventory-snapshots/2026/04/
├── landing/delivery-receipts/2026/04/
├── landing/weather/2026/04/
├── landing/maintenance/
├── curated/trip-segments/
├── curated/delivery-metrics/
├── analytics/route-optimization/
└── analytics/fuel-efficiency/
```

In [ ]:
%%bash
# Create the GreenRoute data lake directory structure
docker exec hadoop-namenode bash -c '
    echo "Creating GreenRoute data lake structure..."

    # Landing zone — GPS telemetry partitioned by day/hour for 7 days
    for day in $(seq -w 1 7); do
        for hour in $(seq -w 0 23); do
            hdfs dfs -mkdir -p /greenroute-lake/landing/gps-telemetry/2026/04/$day/$hour
        done
    done

    # Landing zone — other sources
    hdfs dfs -mkdir -p /greenroute-lake/landing/inventory-snapshots/2026/04
    hdfs dfs -mkdir -p /greenroute-lake/landing/delivery-receipts/2026/04
    hdfs dfs -mkdir -p /greenroute-lake/landing/weather/2026/04
    hdfs dfs -mkdir -p /greenroute-lake/landing/maintenance

    # Curated zone
    hdfs dfs -mkdir -p /greenroute-lake/curated/trip-segments
    hdfs dfs -mkdir -p /greenroute-lake/curated/delivery-metrics

    # Analytics zone
    hdfs dfs -mkdir -p /greenroute-lake/analytics/route-optimization
    hdfs dfs -mkdir -p /greenroute-lake/analytics/fuel-efficiency

    # Archive zone (for data lifecycle management)
    hdfs dfs -mkdir -p /greenroute-lake/archive/gps-telemetry

    echo ""
    echo "=== GreenRoute Data Lake Structure ==="
    hdfs dfs -ls -R /greenroute-lake | head -50
    echo "... (truncated — showing first 50 entries)"
    echo ""
    echo "Total directories created: $(hdfs dfs -ls -R /greenroute-lake | wc -l)"
'

### Step 3.2: Download & Load GPS Telemetry

The course public repository contains pre-generated GPS data for **50 trucks** with **50,000 total readings**, partitioned into **168 hourly files** (7 days x 24 hours). Each partition is a separate Avro file stored at `data/greenroute/gps-telemetry/day_DD/hour_HH/gps_data.avro`.

**Format choice: Avro for GPS telemetry.**

GreenRoute lands GPS in **Apache Avro** format because:

- **Massive size savings**: Each JSON GPS record is ~250 bytes; the same record in Avro compresses to ~80 bytes. At **170 million events/day**, that is roughly **1 TB/day saved** versus JSON.
- **Schema enforcement at write time**: Bad records can't sneak in. The Avro writer rejects anything that doesn't match the schema, so downstream consumers don't have to defend against malformed records.
- **Schema evolution**: When new truck models add sensors (e.g., tire pressure, cabin temperature), Avro handles new fields gracefully -- old readers skip unknown fields, new readers get defaults for missing ones.
- **Kafka-ready**: GPS data often flows through Kafka before landing in HDFS. Avro is Kafka's native serialization format with schema registry support, so the same payloads we land here will flow through Kafka in Week 10.

The schema lives in a sidecar `.avsc` file alongside the data: `data/greenroute/gps-telemetry/gps_reading.avsc`. Avro is binary -- you cannot `cat` it like JSON, but Spark reads it natively (we will see that in Week 9). See `data/generate_all.py` in the data-generation script for how the files were produced.

In [ ]:
%%bash
# Download pre-generated GPS telemetry partitions from the course public class repo
# and upload to HDFS. The GPS files are Avro (binary, schema-embedded) -- the right
# format for high-volume telemetry. We will read them with Spark in Week 9.
docker exec hadoop-namenode bash -c '
    REPO_BASE="https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main/week08/data"

    echo "Downloading 168 GPS telemetry partition files (7 days x 24 hours)..."
    echo ""

    downloaded=0
    for day in 01 02 03 04 05 06 07; do
        for hour in 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23; do
            mkdir -p /tmp/gps_partitions/${day}/${hour}
            curl -sL "$REPO_BASE/greenroute/gps-telemetry/day_${day}/hour_${hour}/gps_data.avro" \
                -o /tmp/gps_partitions/${day}/${hour}/gps_data.avro
            downloaded=$((downloaded + 1))
        done
        echo "  Day $day: downloaded 24 hourly partitions"
    done

    echo ""
    echo "Downloaded $downloaded Avro partition files."
    echo ""
    echo "Sample partition file sizes:"
    ls -lh /tmp/gps_partitions/01/08/gps_data.avro 2>/dev/null
    ls -lh /tmp/gps_partitions/03/14/gps_data.avro 2>/dev/null
    echo ""
    echo "Schema (from the .avsc sidecar in the same directory of the data repo):"
    curl -sL "$REPO_BASE/greenroute/gps-telemetry/gps_reading.avsc" | head -30
    echo ""
    echo "(Avro is a binary format -- you cannot cat it like JSON. We will read"
    echo " these files with Spark in Week 9 using spark.read.format(\"avro\").)"
'


In [ ]:
%%bash
# Upload GPS telemetry partitions to HDFS
docker exec hadoop-namenode bash -c '
    echo "Uploading GPS telemetry partitions to HDFS..."

    uploaded=0
    for day in 01 02 03 04 05 06 07; do
        for hour in 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23; do
            src="/tmp/gps_partitions/${day}/${hour}/gps_data.avro"
            if [ -f "$src" ]; then
                hdfs dfs -put "$src" /greenroute-lake/landing/gps-telemetry/2026/04/$day/$hour/
                uploaded=$((uploaded + 1))
            fi
        done
    done

    echo "Uploaded $uploaded partition files."
    echo ""

    echo "=== Hourly partitions for Day 01 ==="
    hdfs dfs -ls /greenroute-lake/landing/gps-telemetry/2026/04/01/ 2>/dev/null | head -10
    echo ""

    echo "=== Sample: contents of one hourly partition ==="
    hdfs dfs -ls -h /greenroute-lake/landing/gps-telemetry/2026/04/01/08/ 2>/dev/null
    echo ""

    echo "=== Note ==="
    echo "Avro is a binary format -- we cannot cat or pretty-print these files."
    echo "Spark will read them natively in Week 9 with spark.read.format(\"avro\")."
'


**Key takeaway:** By partitioning GPS data into `/year/month/day/hour/` directories, a query like "show me all truck positions on April 1st between 2 PM and 3 PM" only reads `/2026/04/01/14/` — it never touches the other 167 hourly partitions. This is called **partition pruning** and it is essential for performance at scale.

### Step 3.3: Download & Load Delivery Receipts (Nested JSON)

Delivery receipts are deeply nested documents — each delivery contains a list of items, customer feedback, and proof-of-delivery metadata. This is a natural fit for JSON and would be awkward to model in a relational database.

The pre-generated file contains 5,000 delivery receipts with nested item arrays, signature flags, and customer ratings. See `data/generate_all.py` in the repo for the generation script.

In [ ]:
%%bash
# Download pre-generated delivery receipts from the course repo
docker exec hadoop-namenode bash -c '
    REPO_BASE="https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main/week08/data"

    echo "Downloading 5,000 delivery receipts..."
    curl -sL "$REPO_BASE/greenroute/delivery-receipts/delivery_receipts.json" -o /tmp/delivery_receipts.json

    echo ""
    echo "File size:"
    ls -lh /tmp/delivery_receipts.json
    echo ""
    echo "Record count: $(wc -l < /tmp/delivery_receipts.json) records"
    echo ""
    echo "Sample record (pretty-printed):"
    head -1 /tmp/delivery_receipts.json | python3 -m json.tool
'

In [ ]:
%%bash
# Upload delivery receipts to HDFS
docker exec hadoop-namenode bash -c '
    hdfs dfs -put /tmp/delivery_receipts.json /greenroute-lake/landing/delivery-receipts/2026/04/

    echo "=== Delivery receipts uploaded ==="
    hdfs dfs -ls -h /greenroute-lake/landing/delivery-receipts/2026/04/
    echo ""
    echo "Notice the nested structure: details.items is an array of objects."
    echo "In a relational DB, you would need a separate delivery_items table"
    echo "with a foreign key back to deliveries. Schema-on-read avoids this complexity."
'

### Step 3.4: Download & Load Weather Data (JSON)

Weather data is essential for route optimization — rain, snow, and fog all affect delivery times and fuel efficiency. GreenRoute collects hourly weather readings for 20 geographic zones.

The pre-generated file contains 3,360 hourly weather records (20 zones x 7 days x 24 hours). See `data/generate_all.py` in the repo for the generation script.

In [ ]:
%%bash
# Download pre-generated weather data from the course repo
docker exec hadoop-namenode bash -c '
    REPO_BASE="https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main/week08/data"

    echo "Downloading weather data (20 zones x 7 days x 24 hours)..."
    curl -sL "$REPO_BASE/greenroute/weather/weather_data.json" -o /tmp/weather_data.json

    echo ""
    echo "File size:"
    ls -lh /tmp/weather_data.json
    echo ""
    echo "Total readings: $(wc -l < /tmp/weather_data.json)"
    echo ""
    echo "Sample records:"
    head -2 /tmp/weather_data.json | python3 -m json.tool
'

In [ ]:
%%bash
# Upload weather data to HDFS
docker exec hadoop-namenode bash -c '
    hdfs dfs -put /tmp/weather_data.json /greenroute-lake/landing/weather/2026/04/

    echo "=== Weather data uploaded ==="
    hdfs dfs -ls -h /greenroute-lake/landing/weather/2026/04/
'

### Step 3.5: Capacity Planning Exercise

One of the most important questions for a data engineer: **How much storage will we need?**

Let’s calculate the storage requirements for GreenRoute at production scale (2,000 trucks) and project out over time. We will compare **JSON vs Avro** side by side to see the impact of format choice.

In [ ]:
import math

print("=" * 70)
print("    GREENROUTE CAPACITY PLANNING — GPS TELEMETRY")
print("=" * 70)
print()

# Known parameters
trucks = 2000
readings_per_truck_per_day = 8640      # 1 reading every 10 seconds
block_size_mb = 128

# Format-specific sizes
avg_json_bytes = 250       # ~250 bytes per JSON record (with field names)
avg_avro_bytes = 80        # ~80 bytes per Avro record (binary, schema-in-header)

# Calculate daily volume
daily_readings = trucks * readings_per_truck_per_day

json_daily_bytes = daily_readings * avg_json_bytes
json_daily_gb = json_daily_bytes / (1024 ** 3)

avro_daily_bytes = daily_readings * avg_avro_bytes
avro_daily_gb = avro_daily_bytes / (1024 ** 3)

savings_gb = json_daily_gb - avro_daily_gb

print(f"Fleet size:              {trucks:,} trucks")
print(f"Readings/truck/day:      {readings_per_truck_per_day:,}")
print(f"Daily readings:          {daily_readings:,}")
print()

header_col = ""
json_col = "JSON"
avro_col = "Avro"
save_col = "Savings"
json_sub = "(~250 B/rec)"
avro_sub = "(~80 B/rec)"
sep = "-"

print(f"{header_col:<18s} {json_col:>14s} {avro_col:>14s} {save_col:>14s}")
print(f"{header_col:<18s} {json_sub:>14s} {avro_sub:>14s} {header_col:>14s}")
print(f"{sep*18} {sep*14} {sep*14} {sep*14}")

label = "Bytes/record"
print(f"{label:<18s} {avg_json_bytes:>14,d} {avg_avro_bytes:>14,d} {avg_json_bytes - avg_avro_bytes:>14,d}")

label = "Daily volume"
print(f"{label:<18s} {json_daily_gb:>13.1f}G {avro_daily_gb:>13.1f}G {savings_gb:>13.1f}G")

json_blocks = math.ceil(json_daily_gb * 1024 / block_size_mb)
avro_blocks = math.ceil(avro_daily_gb * 1024 / block_size_mb)
label = "Daily blocks"
print(f"{label:<18s} {json_blocks:>14,d} {avro_blocks:>14,d}")
print()

print("-" * 70)
print("STORAGE PROJECTIONS WITH REPLICATION (RF=2)")
print("-" * 70)
print()

h1 = "Time Period"
h2 = "JSON (RF=2)"
h3 = "Avro (RF=2)"
h4 = "Savings"
print(f"{h1:<18s} {h2:>14s} {h3:>14s} {h4:>14s}")
print(f"{sep*18} {sep*14} {sep*14} {sep*14}")

rf = 2
for label, days in [("1 day", 1), ("1 week", 7), ("1 month", 30), ("1 quarter", 90), ("1 year", 365)]:
    json_total = json_daily_gb * days * rf
    avro_total = avro_daily_gb * days * rf
    saved = json_total - avro_total

    def fmt(gb):
        if gb < 1024:
            return f"{gb:.1f} GB"
        else:
            return f"{gb/1024:.1f} TB"

    print(f"{label:<18s} {fmt(json_total):>14s} {fmt(avro_total):>14s} {fmt(saved):>14s}")

print()
print("-" * 70)
print("KEY INSIGHT: Avro saves ~1 TB/day vs JSON at production scale.")
yearly_saved_tb = (savings_gb * 365 * rf) / 1024
print(f"Over 1 year with RF=2, that is {yearly_saved_tb:.0f} TB of physical storage saved.")
print("This is why format choice matters at scale — it directly impacts")
print("hardware costs, network bandwidth, and query performance.")
print("-" * 70)

In [ ]:
%%bash
# Set replication strategies for GreenRoute
docker exec hadoop-namenode bash -c '
    echo "=== Setting Replication Factors for GreenRoute ==="
    echo ""

    # GPS telemetry: RF=2 (high volume, re-collectable)
    echo "Setting RF=2 for GPS telemetry (high volume)..."
    hdfs dfs -setrep -R 2 /greenroute-lake/landing/gps-telemetry/
    echo ""

    # Delivery receipts: RF=3 (business critical)
    echo "Setting RF=3 for delivery receipts (business critical)..."
    hdfs dfs -setrep -R 3 /greenroute-lake/landing/delivery-receipts/
    echo ""

    # Weather: keep default RF=2
    echo "Weather data: keeping cluster default RF=2"
    echo ""

    echo "=== Verification ==="
    echo "GPS telemetry (sample):"
    sample_gps=$(hdfs dfs -ls -R /greenroute-lake/landing/gps-telemetry/ 2>/dev/null | grep "gps_data.avro" | head -1 | awk "{print \$NF}")
    if [ -n "$sample_gps" ]; then
        hdfs dfs -stat "  File: %n | Replication: %r" $sample_gps
    fi
    echo ""
    echo "Delivery receipts:"
    hdfs dfs -stat "  File: %n | Replication: %r" /greenroute-lake/landing/delivery-receipts/2026/04/delivery_receipts.json
    echo ""
    echo "Weather data:"
    hdfs dfs -stat "  File: %n | Replication: %r" /greenroute-lake/landing/weather/2026/04/weather_data.json
'

### Step 3.6: Data Lifecycle Management

Data does not stay "hot" forever. After a week, GPS telemetry becomes historical — still valuable for long-term trend analysis, but not needed for real-time routing decisions. GreenRoute can **archive** old data and **reduce replication** to save storage.

**Storage tiering strategy:**

| Tier | Age | Replication | Use Case |
|------|-----|-------------|----------|
| **Hot** | 0-7 days | RF=2 | Active routing, real-time dashboards |
| **Warm** | 8-90 days | RF=1 | Historical analysis, trend reports |
| **Cold** | 90+ days | RF=1 or offload to S3 | Compliance, long-term archival |

In [ ]:
%%bash
# Demonstrate data lifecycle: archive old GPS data and reduce replication
docker exec hadoop-namenode bash -c '
    echo "=== DATA LIFECYCLE MANAGEMENT ==="
    echo ""

    # Step 1: Show current state of Day 01 GPS data
    echo "BEFORE: Day 01 GPS data in landing zone"
    day01_size=$(hdfs dfs -du -s /greenroute-lake/landing/gps-telemetry/2026/04/01 2>/dev/null | awk "{print \$1}")
    echo "  Logical size: $(hdfs dfs -du -s -h /greenroute-lake/landing/gps-telemetry/2026/04/01 2>/dev/null | awk "{print \$1, \$2}")"
    sample_file=$(hdfs dfs -ls -R /greenroute-lake/landing/gps-telemetry/2026/04/01 2>/dev/null | grep "gps_data.avro" | head -1 | awk "{print \$NF}")
    if [ -n "$sample_file" ]; then
        echo "  Replication factor: $(hdfs dfs -stat %r $sample_file)"
    fi
    echo ""

    # Step 2: Copy Day 01 to archive
    echo "Archiving Day 01 GPS data..."
    hdfs dfs -mkdir -p /greenroute-lake/archive/gps-telemetry/2026/04/01
    hdfs dfs -cp /greenroute-lake/landing/gps-telemetry/2026/04/01/* /greenroute-lake/archive/gps-telemetry/2026/04/01/
    echo "  Copied to archive."
    echo ""

    # Step 3: Reduce replication on archived data to RF=1
    echo "Reducing replication on archived data to RF=1..."
    hdfs dfs -setrep -R 1 /greenroute-lake/archive/gps-telemetry/2026/04/01/
    echo ""

    # Step 4: Remove from landing zone (in production, only after verification)
    echo "Removing Day 01 from landing zone (simulating cleanup)..."
    hdfs dfs -rm -r /greenroute-lake/landing/gps-telemetry/2026/04/01/
    echo ""

    # Step 5: Show the savings
    echo "AFTER: Day 01 GPS data in archive zone"
    archive_sample=$(hdfs dfs -ls -R /greenroute-lake/archive/gps-telemetry/2026/04/01 2>/dev/null | grep "gps_data.avro" | head -1 | awk "{print \$NF}")
    if [ -n "$archive_sample" ]; then
        echo "  Replication factor: $(hdfs dfs -stat %r $archive_sample)"
    fi
    echo "  Archive size: $(hdfs dfs -du -s -h /greenroute-lake/archive/gps-telemetry/2026/04/01 2>/dev/null | awk "{print \$1, \$2}")"
    echo ""

    echo "STORAGE SAVINGS:"
    echo "  Landing zone (RF=2): data stored 2x"
    echo "  Archive zone  (RF=1): data stored 1x"
    echo "  Savings: 50% reduction in physical storage for archived data"
    echo ""
    echo "In production, this lifecycle policy would run automatically."
    echo "Data older than 7 days moves to archive; older than 90 days"
    echo "may be deleted or moved to cold storage (e.g., S3 Glacier)."
'

### Step 3.7: Lessons — The Avro-to-Parquet Pipeline Pattern

Across both business scenarios, a common pattern emerges for how data flows through the lake:

```
Source (devices/trucks) --> Kafka --> Avro (landing zone) --> Spark ETL --> Parquet (curated/analytics)
```

**Why Avro for the landing zone (ingest)?**

- **Row-oriented**: Optimized for writing complete records one at a time (exactly what streaming produces)
- **Schema evolution**: New fields from upgraded devices or trucks are handled gracefully
- **Kafka-native**: Avro + Schema Registry is the de facto standard for Kafka serialization
- **Compact**: 3x smaller than JSON, reducing network and storage costs during high-volume ingest

**Why Parquet for curated/analytics?**

- **Columnar**: If a query only needs `truck_id` and `speed_mph` from a 9-column GPS record, Parquet reads just those 2 columns — skipping 7 columns entirely
- **Predicate pushdown**: Filters like `WHERE speed_mph > 60` can be evaluated at the storage layer, skipping entire row groups
- **Compression**: Columnar layout compresses extremely well (similar values grouped together)

The transformation from Avro to Parquet happens in the **curated zone** using Spark (which we will cover in the coming weeks). This is the **ELT pattern** from the lecture: load raw data first, transform it later.

---

## Part 4: Reflection Questions

Think through the following questions. These connect today’s HDFS hands-on work to broader data engineering concepts that we will build on in the coming weeks.

---

**Question 1:** Compare the two scenarios: which company benefits more from Avro and why? Consider data volume, schema variability, and the ingest pattern in your answer.

---

**Question 2:** HealthPulse device readings and GreenRoute GPS telemetry both use Avro (in production), but for different primary reasons. What are the distinct motivations for each? Think about what makes each data source challenging.

---

**Question 3:** If GreenRoute adds real-time truck tracking via Kafka (streaming GPS events to a dashboard), how does choosing Avro for the landing zone help with that transition? What would need to change if the landing zone used CSV?

---

**Question 4:** Both companies need to **join data across sources** — HealthPulse needs to join patients with claims to find readmission patterns; GreenRoute needs to join GPS data with delivery receipts to calculate actual vs. estimated delivery times. Why can’t they do this in HDFS alone? What tool would they need? *(Hint: this previews what we will cover with Spark.)*

---

## Summary

In this companion notebook, you explored HDFS internals and built two production-style data lakes:

| | HealthPulse | GreenRoute |
|---|---|---|
| **Industry** | Healthcare | Logistics |
| **Data sources** | Patient records (CSV), device readings (Avro/JSON), insurance claims (CSV) | GPS telemetry (Avro/JSON), delivery receipts (nested JSON), weather (JSON) |
| **Key challenge** | Schema variation across hospitals | High-volume time-series data |
| **HDFS concept demonstrated** | Schema-on-read, replication for compliance | Temporal partitioning, data lifecycle |
| **Replication strategy** | RF=3 for patients (HIPAA), RF=2 for devices | RF=3 for receipts (business-critical), RF=2 for GPS |
| **Why Avro?** | Schema evolution for new device types, compact binary | Massive volume savings (~1 TB/day), Kafka-ready for streaming |
| **Curated format** | Parquet (columnar analytics) | Parquet (columnar analytics) |

### Key HDFS Concepts Practiced

1. **Block storage and replication** — How files split into 128MB blocks and replicate across DataNodes
2. **Rack-aware placement** — NameNode decides placement using topology map; client-to-DataNode pipeline for writes
3. **Read path** — Client gets metadata from NameNode, reads data directly from DataNodes
4. **HDFS CLI** — ls, mkdir, put, get, cat, stat, du, rm, setrep, dfsadmin, fsck
5. **Directory design** — Landing / Curated / Analytics zones
6. **Schema-on-read** — Accept heterogeneous data as-is, reconcile at query time
7. **Temporal partitioning** — Organize time-series data by date/hour for efficient queries
8. **Replication strategies** — Different RF for different data criticality levels
9. **File format trade-offs** — CSV for flat data, JSON for nested, Avro for ingest, Parquet for analytics
10. **Capacity planning** — Projecting storage needs at production scale (JSON vs Avro comparison)
11. **Data lifecycle** — Archiving old data with reduced replication to save storage

### What’s Next

You have data in HDFS, but you cannot query it with SQL or join across sources directly. In the coming weeks, we will use **Apache Spark** to read from this data lake, perform transformations, run aggregations, and build analytics pipelines on top of the foundation you built today.